In [1]:
# Import necessary libraries
import gmsh
import math
import os

In [ ]:
class Mesh_Generator:
    def __init__(self, domain_length, domain_height, filename, hx_objects, mesh_params):
        self.length = domain_length
        self.height = domain_height
        self.filename = filename
        # hx_objects is a list of dictionaries, where each dictionary contains the type and position properties of an obstruction
        self.hx_objects = hx_objects
        # mesh_params is a dictionary containing parameters for mesh generation, such as 'mesh_min' and 'mesh_max'
        self.mesh_params = mesh_params

    def set_filename(self, new_filename):
        self.filename = new_filename

    def add_hx_objects(self, new_hx_objects):
        for obj in new_hx_objects:
            self.hx_objects.append(obj)
        
    def create_custom_obstruction(self, point_coords):
        # Use the provided coordinates to create points in Gmsh
        n_points = len(point_coords)
        points = []
        for coords in point_coords:
            point = gmsh.model.occ.addPoint(coords[0], coords[1], 0)
            points.append(point)
        
        # Connect the points with lines to form the geometry of the obstruction
        lines = []
        for i in range(n_points):
            p1 = points[i]
            p2 = points[(i + 1) % n_points]  # Wrap around to connect the last point to the first
            line = gmsh.model.occ.addLine(p1, p2)
            lines.append(line)

        # Close the loop to create a surface
        cl = gmsh.model.occ.addCurveLoop(lines)
        surface = gmsh.model.occ.addPlaneSurface([cl])
        return surface
    
    def create_polygon_obstruction(self, xc, yc, r, n_pts):
        points = []
        lines = []

        # Create points around the circle
        for i in range(n_pts):
            theta = 2 * math.pi * i / n_pts
            x = xc + r * math.cos(theta)
            y = yc + r * math.sin(theta)
            p = gmsh.model.occ.addPoint(x, y, 0)
            points.append(p)

        # Create lines between consecutive points
        for i in range(n_pts):
            p1 = points[i]
            p2 = points[(i + 1) % n_pts]
            line = gmsh.model.occ.addLine(p1, p2)
            lines.append(line)

        # Create closed loop and surface
        curve_loop = gmsh.model.occ.addCurveLoop(lines)
        surface = gmsh.model.occ.addPlaneSurface([curve_loop])
        return surface
    
    def create_rectangle_obstruction(self, center, width, height):
        # Calculate the lower-left corner of the rectangle and create the rectangle using Gmsh's built-in function
        return gmsh.model.occ.addRectangle(center[0] - width / 2, center[1] - height / 2, 0, width, height)
    
    def create_disk_obstruction(self, center, radius):
        # Create a disk obstruction using Gmsh's built-in function
        disk = gmsh.model.occ.addDisk(center[0], center[1], 0, radius, radius)
        return disk

    def generate_mesh(self):
        # Initialize Gmsh and create a new model
        gmsh.initialize()
        gmsh.model.add("heat_exchanger")

        # Use OpenCASCADE kernel
        gmsh.model.occ

        # Define the domain geometry
        rect = gmsh.model.occ.addRectangle(0, -self.height/2, 0, self.length, self.height)
        gmsh.model.occ.synchronize()

        # Add obstructions to the geometry
        obstructions = []
        for obj in self.hx_objects:
            if obj['type'] == 'polygon':
                obstruction = self.create_polygon_obstruction(obj['center'][0], obj['center'][1], obj['radius'], obj['n_pts'])
            elif obj['type'] == 'rectangle':
                obstruction = self.create_rectangle_obstruction(obj['center'], obj['width'], obj['height'])
            elif obj['type'] == 'disk':
                obstruction = self.create_disk_obstruction(obj['center'], obj['radius'])
            elif obj['type'] == 'custom':
                obstruction = self.create_custom_obstruction(obj['coords'])
            obstructions.append(obstruction)
        gmsh.model.occ.synchronize()

        # Boolean cut
        cut = gmsh.model.occ.cut(
            [(2, rect)],
            [(2, s) for s in obstructions],
            removeObject=True,
            removeTool=True
        )
        gmsh.model.occ.synchronize()

        # Extract surfaces
        outDimTags, _ = cut
        fluid_surfaces = [tag for dim, tag in outDimTags if dim == 2]

        if not fluid_surfaces:
            raise RuntimeError("Boolean cut failed")

        gmsh.model.addPhysicalGroup(2, fluid_surfaces, name="Fluid")

        print("Cut result:", outDimTags)
        print("2D entities:", gmsh.model.getEntities(2))
        
        # Cylinder wall curves
        gmsh.model.occ.synchronize()
        boundaries = gmsh.model.getBoundary(
            [(2, s) for s in fluid_surfaces],
            oriented=False
        )

        obstruction_wall_curves = []
        inlet = []
        outlet = []
        top = []
        bottom = []

        for dim, tag in boundaries:
            com = gmsh.model.occ.getCenterOfMass(dim, tag)
            x, y, _ = com

            if abs(x) < 1e-6:
                inlet.append(tag)
            elif abs(x - self.length) < 1e-6:
                outlet.append(tag)
            elif abs(y - self.height/2) < 1e-6:
                top.append(tag)
            elif abs(y + self.height/2) < 1e-6:
                bottom.append(tag)
            else:
                obstruction_wall_curves.append(tag)

        gmsh.model.addPhysicalGroup(1, obstruction_wall_curves, name="Wall")
        gmsh.model.addPhysicalGroup(1, inlet, name="Inlet")
        gmsh.model.addPhysicalGroup(1, outlet, name="Outlet")
        gmsh.model.addPhysicalGroup(1, top, name="Top")
        gmsh.model.addPhysicalGroup(1, bottom, name="Bottom")
        
        # Set mesh parameters
        gmsh.option.setNumber("Mesh.CharacteristicLengthMin", self.mesh_params['mesh_min'])
        gmsh.option.setNumber("Mesh.CharacteristicLengthMax", self.mesh_params['mesh_max'])
        gmsh.option.setNumber("Mesh.Algorithm", self.mesh_params['mesh_algorithm']) # 8 
        gmsh.option.setNumber("Mesh.RecombineAll", self.mesh_params['mesh_recombine']) # 1
        gmsh.option.setNumber("Mesh.ElementOrder", self.mesh_params['mesh_element_order']) # 2

        # Generate the mesh
        gmsh.model.mesh.generate(2)
        gmsh.write(self.filename)
        gmsh.finalize()

In [27]:
generator = Mesh_Generator(
    domain_length=10.0,
    domain_height=2.0,
    filename="heat_exchanger.msh",
    hx_objects=[
        {'type': 'custom',
         'coords': [(2.0, -0.2), (2.1, -0.2), (2.1, -0.3), (2.2, -0.3), (2.2, -0.4), (2.6, -0.4),
         (2.6, -0.3), (2.7, -0.3), (2.7, -0.2), (2.8, -0.2), (2.8, 0.2), (2.7, 0.2), (2.7, 0.3), (2.6, 0.3), (2.6, 0.4), 
         (2.2, 0.4), (2.2, 0.3), (2.1, 0.3), (2.1, 0.2), (2.0, 0.2)]},
         {'type': 'disk',
          'center': (5.2, 0.0),
          'radius': 0.4},
         {'type': 'custom',
          'coords': [(8.0, 0.0), (8.5, -0.35), (8.5, 0.35)]},
        {'type': 'rectangle',
         'center': (7.0, 0.5),
         'width': 0.5,
         'height': 1.0}
    ],
    mesh_params={
        'mesh_min': 0.005,
        'mesh_max': 0.05,
        'mesh_algorithm': 8,  # Frontal-Delaunay for 2D
        'mesh_recombine': 1,  # Recombine into quadrangles
        'mesh_element_order': 2,  # Use second-order elements
    }
)
generator.generate_mesh()

Cut result: [(2, 1)]                                                                                                                 
2D entities: [(2, 1)]
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 5 (Line)
Info    : [ 10%] Meshing curve 6 (Line)
Info    : [ 10%] Meshing curve 7 (Line)
Info    : [ 10%] Meshing curve 8 (Line)
Info    : [ 20%] Meshing curve 9 (Line)
Info    : [ 20%] Meshing curve 10 (Line)
Info    : [ 20%] Meshing curve 11 (Line)
Info    : [ 30%] Meshing curve 12 (Line)
Info    : [ 30%] Meshing curve 13 (Line)
Info    : [ 30%] Meshing curve 14 (Line)
Info    : [ 40%] Meshing curve 15 (Line)
Info    : [ 40%] Meshing curve 16 (Line)
Info    : [ 40%] Meshing curve 17 (Line)
Info    : [ 50%] Meshing curve 18 (Line)
Info    : [ 50%] Meshing curve 19 (Line)
Info    : [ 50%] Meshing curve 20 (Line)
Info    : [ 60%] Meshing curve 21 (Line)
Info    : [ 60%] Meshing curve 22 (Line)
Info    : [ 60%] Meshing curve 23 (Line)
Info    : [ 60%] Meshing curve 24 (Line)
Info 

In [7]:
# Main function to generate the computational domain
def generate_cylinder_array_mesh(
    array_height=5,
    array_width=7,
    l_start=2,
    l_after=10,
    D=1,
    SL=1.5,
    ST=1.5,
    mesh_min=0.05,
    mesh_max=0.1,
    filename="cyl.msh"
):

    gmsh.initialize()
    gmsh.model.add("cylinder_array")

    # Use OpenCASCADE kernel
    gmsh.model.occ

    # --------------------------------------------------
    # DOMAIN SIZE
    # --------------------------------------------------
    l_x = l_start + D + SL * (array_width - 1) + l_after
    l_y = ST * array_height

    # --------------------------------------------------
    # CREATE RECTANGLE DOMAIN
    # --------------------------------------------------
    rect = gmsh.model.occ.addRectangle(-l_start, -l_y/2, 0, l_x, l_y)

    cylinder_surfaces = []

    # --------------------------------------------------
    # FULL CYLINDERS
    # --------------------------------------------------
    for i in range(array_width):
        for j in range(array_height - (1 + (i % 2))):

            x = D/2 + SL * i
            y = (l_y - ST)/2 - (ST * j) - (i % 2) * ST/2

            poly = gmsh.model.occ.addDisk(x, y, 0, D/2, D/2)
            cylinder_surfaces.append(poly)

    # --------------------------------------------------
    # HALF CYLINDERS (top and bottom staggered columns)
    # --------------------------------------------------
    half_surfaces = []

    for i in range(1, array_width, 2):

        x_center = D/2 + SL * i

        # Top half
        top_disk = gmsh.model.occ.addDisk(x_center, l_y/2, 0, D/2, D/2)
        top_cut = gmsh.model.occ.addRectangle(
            x_center - D, l_y/2, 0, 2*D, D
        )
        top_half = gmsh.model.occ.cut(
            [(2, top_disk)],
            [(2, top_cut)],
            removeTool=True
        )[0]

        for dim, tag in top_half:
            half_surfaces.append(tag)

        # Bottom half
        bot_disk = gmsh.model.occ.addDisk(x_center, -l_y/2, 0, D/2, D/2)
        bot_cut = gmsh.model.occ.addRectangle(
            x_center - D, -l_y/2 - D, 0, 2*D, D
        )
        bot_half = gmsh.model.occ.cut(
            [(2, bot_disk)],
            [(2, bot_cut)],
            removeTool=True
        )[0]

        for dim, tag in bot_half:
            half_surfaces.append(tag)

    # --------------------------------------------------
    # BOOLEAN DIFFERENCE (subtract cylinders from domain)
    # --------------------------------------------------
    gmsh.model.occ.synchronize()

    if len(half_surfaces) > 0:    
        cut = gmsh.model.occ.cut(
            [(2, rect)],
            [(2, s) for s in cylinder_surfaces + half_surfaces],
            removeTool=True
        )
    else:
        cut = gmsh.model.occ.cut(
            [(2, rect)],
            [(2, s) for s in cylinder_surfaces],
            removeTool=True
        )

    gmsh.model.occ.synchronize()

    fluid_surface = cut[0][0][1]

    # --------------------------------------------------
    # PHYSICAL GROUPS
    # --------------------------------------------------
    gmsh.model.addPhysicalGroup(2, [fluid_surface], name="Fluid")

    # Cylinder wall curves
    gmsh.model.occ.synchronize()
    boundaries = gmsh.model.getBoundary([(2, fluid_surface)], oriented=False)

    cylinder_wall_curves = []
    inlet = []
    outlet = []
    top = []
    bottom = []

    for dim, tag in boundaries:
        com = gmsh.model.occ.getCenterOfMass(dim, tag)

        x, y, _ = com

        if abs(x + l_start) < 1e-6:
            inlet.append(tag)
        elif abs(x - (l_x - l_start)) < 1e-6:
            outlet.append(tag)
        elif abs(y - l_y/2) < 1e-6:
            top.append(tag)
        elif abs(y + l_y/2) < 1e-6:
            bottom.append(tag)
        else:
            cylinder_wall_curves.append(tag)

    gmsh.model.addPhysicalGroup(1, cylinder_wall_curves, name="CylinderWalls")
    gmsh.model.addPhysicalGroup(1, inlet, name="Inlet")
    gmsh.model.addPhysicalGroup(1, outlet, name="Outlet")
    gmsh.model.addPhysicalGroup(1, top, name="Top")
    gmsh.model.addPhysicalGroup(1, bottom, name="Bottom")

    # --------------------------------------------------
    # MESH SETTINGS
    # --------------------------------------------------
    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", mesh_min)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", mesh_max)

    gmsh.option.setNumber("Mesh.Algorithm", 8)
    gmsh.option.setNumber("Mesh.RecombineAll", 1)
    gmsh.option.setNumber("Mesh.ElementOrder", 2)

    gmsh.model.mesh.generate(2)
    gmsh.write(filename)

    gmsh.finalize()

    print(f"Mesh written to {filename}")

In [8]:
generate_cylinder_array_mesh(array_height=1, array_width=1, filename="cyls.msh")

Error   : Difference failed - BOPAlgo_AlertTooFewArguments 


Exception: Difference failed - BOPAlgo_AlertTooFewArguments 